# Gym environment

Testing gym environments.

In [1]:
import gymnasium as gym


In [2]:
env =gym.make("LunarLander-v3", render_mode="human")

In [3]:
observation, info = env.reset(seed=42)

In [4]:
for _ in range(1000):
    action = env.action_space.sample()

    observation, reward, terminated, truncated, info = env.step(action)

    if terminated or truncated:
        observation, info = env.reset()

env.close()

In [ ]:
env.close()

: 

## Petting zoo

In [ ]:
from pettingzoo.classic import chess_v6 as kaz

In [19]:
env = kaz.env(render_mode="human")
env.reset(seed=42)

for agent in env.agent_iter():
    observation, reward, termination, truncation, info = env.last()
    print(observation.keys())
    if termination or truncation:
        action = None
    else:
        action = env.action_space(agent).sample()
        action = [i for i in range(len(observation["action_mask"])) if observation["action_mask"][i] == 1][0]
    
    env.step(action)
    print(info)


dict_keys(['observation', 'action_mask'])
{}
dict_keys(['observation', 'action_mask'])
{}
dict_keys(['observation', 'action_mask'])
{}
dict_keys(['observation', 'action_mask'])
{}
dict_keys(['observation', 'action_mask'])
{}
dict_keys(['observation', 'action_mask'])
{}
dict_keys(['observation', 'action_mask'])
{}
dict_keys(['observation', 'action_mask'])
{}
dict_keys(['observation', 'action_mask'])
{}
dict_keys(['observation', 'action_mask'])
{}
dict_keys(['observation', 'action_mask'])
{}
dict_keys(['observation', 'action_mask'])
{'legal_moves': []}
dict_keys(['observation', 'action_mask'])
{'legal_moves': []}


In [ ]:
env.close()

In [ ]:
from stable_baselines3 import HerReplayBuffer, DDPG, DQN, SAC, TD3
from stable_baselines3.her.goal_selection_strategy import GoalSelectionStrategy
from stable_baselines3.common.envs import BitFlippingEnv
import gymnasium as gym

model_class = DQN  # works also with SAC, DDPG and TD3

env = gym.make("CartPole-v1", render_mode="rgb_array")

# Available strategies (cf paper): future, final, episode
goal_selection_strategy = "future" # equivalent to GoalSelectionStrategy.FUTURE

# Initialize the model
model = model_class(
    "MultiInputPolicy",
    env,
    replay_buffer_class=HerReplayBuffer,
    # Parameters for HER
    replay_buffer_kwargs=dict(
        n_sampled_goal=4,
        goal_selection_strategy=goal_selection_strategy,
    ),
    verbose=1,
)

# Train the model
model.learn(1000)

model.save("./her_bit_env")
# Because it needs access to `env.compute_reward()`
# HER must be loaded with the env
model = model_class.load("./her_bit_env", env=env)

obs, info = env.reset()
for _ in range(100):
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, _ = env.step(action)
    model.render("human")
    if terminated or truncated:
        obs, info = env.reset()

In [ ]:
from stable_baselines3 import HerReplayBuffer, DDPG, DQN, SAC, TD3, A2C
from stable_baselines3.her.goal_selection_strategy import GoalSelectionStrategy 
import gymnasium as gym


model_class = A2C

env = gym.make("CartPole-v1", render_mode="rgb_array")

goal_selection_strategy = "future"

model = model_class(
    "MultiInputPolicy",
    env,
    # replay_buffer_class=HerReplayBuffer,
    # # Parameters for HER
    # replay_buffer_kwargs=dict(
    #     n_sampled_goal=4,
    #     goal_selection_strategy=goal_selection_strategy,
    # ),
    verbose=1,
)

model.learn(total_timesteps=10_000)

model.save("./her_bit_env")
model = model_class.load("./her_bit_env", env=env)

vec_env = model.get_env()
obs, info = vec_env.reset(seed=42)

for _ in range(1000):
    action, _state = model.predict(obs, deterministic = True)

    obs, reward, terminated, truncated, info = vec_env.step(action)
    vec_env.render("human")
    if terminated or truncated:
        obs, info = env.reset()

env.close()